# PhysREVE — Seizure Detection (CHB-MIT)

Tests the PhysREVE encoder on **binary seizure detection** using the
[CHB-MIT Scalp EEG Database](https://physionet.org/content/chbmit/1.0.0/) (open access).

**Key differences from the motor-imagery notebook:**
- Binary labels: `0 = interictal` (normal), `1 = ictal` (seizure)
- Bipolar montage → physics-correct bipolar leadfield
- No asymmetry loss (hemispheric lateralisation not applicable here)
- Smaller architecture (limited labelled data per subject)

**Workflow:**
1. Download CHB-MIT patient 1 EDF files (~350 MB total)
2. Build bipolar leadfield
3. Pretrain PhysREVE on unlabelled interictal EEG
4. Fine-tune for seizure detection
5. Evaluate with accuracy, sensitivity, specificity, AUC


In [5]:
import subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo = '/content/PhysREVE'
    if not os.path.exists(repo):
        subprocess.check_call(['git', 'clone', '-q', 'https://github.com/UgoBruzadin/PhysREVE.git', repo])
    else:
        subprocess.check_call(['git', '-C', repo, 'pull', '-q'])
    if repo not in sys.path:
        sys.path.insert(0, repo)
    # Colab already ships torch, numpy, scipy, sklearn, matplotlib, seaborn, tqdm, requests
    # Install only what's missing
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mne>=1.6', 'moabb>=1.0', 'xgboost'])
    print('Colab environment ready.')
else:
    print('Local environment — ensure you ran: pip install -e .')

In [2]:
# ── 1. Imports & setup ───────────────────────────────────────────────────────
import sys, os, warnings
sys.path.insert(0, '.')   # physreve package lives in the same directory

import numpy as np
import torch
import mne
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
)
warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device  : {device}')
print(f'PyTorch : {torch.__version__}')
print(f'MNE     : {mne.__version__}')


Device  : cuda
PyTorch : 2.10.0+cu128
MNE     : 1.12.0


## 1  Download CHB-MIT (Patient chb01)

We download only the 7 files that contain seizures plus 2 seizure-free files
for the unlabelled pretraining corpus.  Total ≈ 350 MB — cached after first run.


In [3]:
from physreve.datasets.chbmit import download_patient, parse_summary

DATA_DIR    = './data/chb-mit'
PATIENT_ID  = 'chb01'

# Files that contain seizures (chb01 has 7 seizures across these files)
SEIZURE_FILES = [
    'chb01_03.edf',  # seizure 2996-3036 s
    'chb01_04.edf',  # seizure 1467-1494 s
    'chb01_15.edf',  # seizure 1732-1772 s
    'chb01_16.edf',  # seizure 1015-1066 s
    'chb01_18.edf',  # seizure 1720-1810 s
    'chb01_21.edf',  # seizure  327-420  s
    'chb01_26.edf',  # seizure 1862-1963 s
]

# Extra seizure-free files for pretraining interictal pool
PRETRAIN_FILES = [
    'chb01_01.edf',
    'chb01_02.edf',
]

print('Downloading CHB-MIT patient 1...')
download_patient(
    patient_id=PATIENT_ID,
    data_dir=DATA_DIR,
    files=SEIZURE_FILES + PRETRAIN_FILES,
)
print('\nAll files ready.')


ModuleNotFoundError: No module named 'physreve'

## 2  Parse Seizure Annotations & Window Data

In [ ]:
from physreve.datasets.chbmit import parse_summary, load_patient_epochs
from pathlib import Path

summary_path = Path(DATA_DIR) / PATIENT_ID / f'{PATIENT_ID}-summary.txt'
seizure_info = parse_summary(str(summary_path))

print('Seizure annotations:')
for fname, seiz in seizure_info.items():
    if seiz:
        for s, e in seiz:
            print(f'  {fname}: {s}–{e} s  ({e-s} s duration)')


In [ ]:
# Extract 4-second windows at 200 Hz
# Ictal:       within seizure (with 2 s buffer at boundaries)
# Interictal:  >= 60 s from any seizure

ictal_X, interictal_X, ch_names = load_patient_epochs(
    data_dir=DATA_DIR,
    patient_id=PATIENT_ID,
    seizure_info=seizure_info,
    window_sec=4.0,
    stride_sec=2.0,
    target_sfreq=200.0,
    ictal_buffer_sec=2.0,
    interictal_gap_sec=60.0,
    max_interictal_per_file=300,
    verbose=True,
)

N_CH  = ictal_X.shape[1]
T_WIN = ictal_X.shape[2]   # 800 samples = 4 s at 200 Hz
print(f'\nChannels ({N_CH}): {ch_names}')
print(f'Window   : {T_WIN} samples ({T_WIN/200:.1f} s at 200 Hz)')


## 3  Build Bipolar Leadfield

CHB-MIT uses a **bipolar montage** (e.g. "FP1-F7").
For the physics forward model:

$$L_{bip}[i, j] = L_{ref}[e_{1,i}, j] - L_{ref}[e_{2,i}, j]$$

where $L_{ref}$ is the standard referential leadfield for the underlying electrodes.


In [ ]:
from physreve.datasets.chbmit import build_leadfield_bipolar

print(f'Building bipolar leadfield for {N_CH} channels...')
L_col_np, L_row_np, src_pos_np, info_ref, elec_xyz_np, valid_mask = \
    build_leadfield_bipolar(ch_names, sfreq=200.0, grid_pos=10.0, verbose=True)

N_SRC = L_col_np.shape[1]
print(f'\nLeadfield ready:')
print(f'  Channels × Sources : {L_col_np.shape}')
print(f'  Channels with valid bipolar mapping: {sum(valid_mask)}/{len(valid_mask)}')

# Move to device
L_col    = torch.tensor(L_col_np,    dtype=torch.float32).to(device)
L_row    = torch.tensor(L_row_np,    dtype=torch.float32).to(device)
ELEC_XYZ = torch.tensor(elec_xyz_np, dtype=torch.float32).to(device)  # (N_CH, 3)


## 4  Datasets & DataLoaders

In [ ]:
from physreve.datasets.chbmit import make_seizure_loaders

pretrain_loader, train_loader, val_loader, test_loader = make_seizure_loaders(
    ictal_X, interictal_X,
    train_frac=0.70,
    val_frac=0.15,
    batch_size=16,
    seed=SEED,
    balance=True,
)

CLASS_NAMES = ['Interictal', 'Ictal']


## 5  PhysREVE Configuration

We use a **smaller architecture** than the BCI IV 2a notebook — the labelled
dataset (few hundred windows) is much smaller than a large motor-imagery corpus.
Regularisation (`dropout=0.25`) and shorter pretraining keep the model from
over-parameterising the limited data.

`d_pos_x + d_pos_y + d_pos_z + d_pos_t` must equal `d_model`.


In [ ]:
from physreve import PhysREVEConfig

CFG = PhysREVEConfig(
    # Smaller transformer — limited labelled data
    d_model      = 128,
    n_heads      = 4,
    n_layers     = 6,
    d_ff         = 512,
    dropout      = 0.25,

    # 50 samples/patch = 0.25 s at 200 Hz
    patch_size   = 50,
    mask_ratio   = 0.75,
    block_t      = 4,
    block_c      = 2,

    n_sources    = N_SRC,

    # Physics weights — keep physics loss; disable asymmetry (not applicable)
    lambda_phys  = 0.15,
    lambda_snr   = 0.05,
    lambda_asym  = 0.0,      # disabled

    leadfield_bias_scale = 0.5,

    # 4D pos enc: 32 × 4 = 128 = d_model
    d_pos_x      = 32,
    d_pos_y      = 32,
    d_pos_z      = 32,
    d_pos_t      = 32,

    exp_name     = '_seizure_chbmit',
)

# Number of patches per window (for block mask)
N_PATCHES = T_WIN // CFG.patch_size
print(f'Config OK: d_model={CFG.d_model}, n_layers={CFG.n_layers}')
print(f'Patches per window: {N_PATCHES}  ({CFG.patch_size} samples each)')
print(f'n_sources (actual from leadfield): {N_SRC}')


## 6  Phase 1 — Pretraining (Physics-Informed MAE)

Pretrain on **unlabelled interictal** windows.  The model learns general EEG
representations with MAE + physics consistency, without seeing any seizure labels.


In [ ]:
from physreve.train import run_pretraining

print('='*60)
print('PHASE 1: Pretraining on unlabelled interictal EEG')
print('='*60)

pretrained_model, pretrain_hist = run_pretraining(
    cfg       = CFG,
    loader    = pretrain_loader,
    L_row     = L_row,
    L_col     = L_col,
    elec_xyz  = ELEC_XYZ,
    device    = device,
    n_epochs  = 20,
    lr        = 3e-4,
    sfreq     = 200.0,
    save_path = f'physreve_pretrained{CFG.exp_name}.pt',
)


In [ ]:
# ── Pretraining loss curves ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
fig.suptitle('PhysREVE Pretraining — Loss Components', fontsize=12)

for ax, key, label, color in zip(
    axes,
    ['mae', 'phys', 'snr'],
    ['MAE reconstruction', 'Physics consistency', 'SNR alignment'],
    ['royalblue', 'darkorange', 'forestgreen'],
):
    ax.plot(pretrain_hist[key], color=color, lw=2)
    ax.set_title(label)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'pretrain_losses{CFG.exp_name}.png', dpi=150)
plt.show()


## 7  Phase 2 — Fine-Tuning for Seizure Detection

Fine-tune with:
- **2 output classes** (interictal / ictal)
- **No asymmetry loss** (`lh_idx=None`)
- Differential LR: frozen encoder for 5 epochs, then unfrozen


In [ ]:
from physreve.train import run_finetuning, run_baseline_finetune

print('='*60)
print('PHASE 2a: Baseline (random init, no pretraining)')
print('='*60)

model_base, hist_base = run_baseline_finetune(
    cfg          = CFG,
    train_loader = train_loader,
    val_loader   = val_loader,
    L_row        = L_row,
    L_col        = L_col,
    elec_xyz     = ELEC_XYZ,
    device       = device,
    n_classes    = 2,
    n_epochs     = 30,
    lr           = 3e-4,
    lh_idx       = None,   # no asymmetry loss
    rh_idx       = None,
)


In [ ]:
print('='*60)
print('PHASE 2b: PhysREVE (physics pretrained)')
print('='*60)

model_phys, hist_phys = run_finetuning(
    pretrained    = pretrained_model,
    cfg           = CFG,
    train_loader  = train_loader,
    val_loader    = val_loader,
    L_col         = L_col,
    elec_xyz      = ELEC_XYZ,
    device        = device,
    n_classes     = 2,
    n_epochs      = 30,
    lr_head       = 1e-3,
    lr_enc        = 1e-4,
    freeze_enc_epochs = 5,
    lh_idx        = None,
    rh_idx        = None,
    label_smoothing = 0.05,
    save_path     = f'physreve_finetune{CFG.exp_name}.pt',
)


## 8  Evaluation

In [ ]:
from physreve.evaluate import evaluate, print_results, compare_models

@torch.no_grad()
def evaluate_with_probs(model, loader, elec_xyz, device):
    """Like evaluate() but also returns class probabilities for AUC."""
    import torch.nn.functional as F
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        xyz = elec_xyz.unsqueeze(0).expand(len(Xb), -1, -1)
        logits, _, _ = model(Xb, xyz)
        probs = F.softmax(logits, dim=-1)
        all_preds.append(logits.argmax(1).cpu())
        all_labels.append(yb.cpu())
        all_probs.append(probs.cpu())
    return (
        torch.cat(all_preds).numpy(),
        torch.cat(all_labels).numpy(),
        torch.cat(all_probs).numpy(),
    )

p_b, l_b, prob_b = evaluate_with_probs(model_base, test_loader, ELEC_XYZ, device)
p_p, l_p, prob_p = evaluate_with_probs(model_phys, test_loader, ELEC_XYZ, device)

acc_b = (p_b == l_b).mean()
acc_p = (p_p == l_p).mean()

# Sensitivity (recall on ictal class) and specificity
def sens_spec(preds, labels):
    cm = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()
    return tp / (tp + fn + 1e-8), tn / (tn + fp + 1e-8)

sens_b, spec_b = sens_spec(p_b, l_b)
sens_p, spec_p = sens_spec(p_p, l_p)

auc_b = roc_auc_score(l_b, prob_b[:, 1])
auc_p = roc_auc_score(l_p, prob_p[:, 1])

print('\n' + '='*65)
print('TEST SET RESULTS')
print('='*65)
print(f'{"Model":<35} {"Acc":>7} {"Sens":>7} {"Spec":>7} {"AUC":>7}')
print('-'*65)
print(f'{"Baseline (random init)":<35} {acc_b*100:>6.1f}% {sens_b*100:>6.1f}% {spec_b*100:>6.1f}% {auc_b:>6.3f}')
print(f'{"PhysREVE (pretrained)":<35} {acc_p*100:>6.1f}% {sens_p*100:>6.1f}% {spec_p*100:>6.1f}% {auc_p:>6.3f}')
print('='*65)
print(f'Δ Accuracy: {(acc_p - acc_b)*100:+.1f}%   Δ AUC: {(auc_p - auc_b):+.3f}')


In [ ]:
# ── Full report ──────────────────────────────────────────────────────────────
print('\n--- PhysREVE Classification Report ---')
print(classification_report(l_p, p_p, target_names=CLASS_NAMES))

print('--- Baseline Classification Report ---')
print(classification_report(l_b, p_b, target_names=CLASS_NAMES))


In [ ]:
# ── Visualisation ────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
fig.suptitle(
    f'PhysREVE Seizure Detection — CHB-MIT Patient 01\n'
    f'Baseline acc={acc_b*100:.1f}%  PhysREVE acc={acc_p*100:.1f}%',
    fontsize=13
)
gs = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)

# (0,0) val accuracy curves
ax = fig.add_subplot(gs[0, 0])
ax.plot(hist_base['val_acc'], label='Baseline', color='slategray', lw=1.5)
ax.plot(hist_phys['val_acc'], label='PhysREVE', color='royalblue',  lw=2)
ax.set_title('Validation Accuracy')
ax.set_xlabel('Epoch')
ax.legend()
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)

# (0,1) ROC curves
ax = fig.add_subplot(gs[0, 1])
fpr_b, tpr_b, _ = roc_curve(l_b, prob_b[:, 1])
fpr_p, tpr_p, _ = roc_curve(l_p, prob_p[:, 1])
ax.plot(fpr_b, tpr_b, color='slategray', lw=1.5, label=f'Baseline  AUC={auc_b:.3f}')
ax.plot(fpr_p, tpr_p, color='royalblue',  lw=2,   label=f'PhysREVE  AUC={auc_p:.3f}')
ax.plot([0,1], [0,1], 'k--', lw=0.8)
ax.set_title('ROC Curve')
ax.set_xlabel('FPR (1 − Specificity)')
ax.set_ylabel('TPR (Sensitivity)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# (0,2) Bar: Accuracy comparison
ax = fig.add_subplot(gs[0, 2])
bars = ax.bar(['Baseline', 'PhysREVE'], [acc_b*100, acc_p*100],
              color=['slategray', 'royalblue'], width=0.5)
ax.axhline(50, color='red', lw=1, linestyle='--', label='Chance (50%)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Test Accuracy')
ax.set_ylim(0, 105)
ax.legend(fontsize=9)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)
ax.grid(alpha=0.3, axis='y')

# (1,0) Confusion matrix — Baseline
ax = fig.add_subplot(gs[1, 0])
cm_b = confusion_matrix(l_b, p_b)
sns.heatmap(cm_b, annot=True, fmt='d', ax=ax, cbar=False,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap='Blues')
ax.set_title('Baseline — Confusion Matrix')
ax.set_ylabel('True')
ax.set_xlabel('Predicted')

# (1,1) Confusion matrix — PhysREVE
ax = fig.add_subplot(gs[1, 1])
cm_p = confusion_matrix(l_p, p_p)
sns.heatmap(cm_p, annot=True, fmt='d', ax=ax, cbar=False,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap='Blues')
ax.set_title('PhysREVE — Confusion Matrix')
ax.set_ylabel('True')
ax.set_xlabel('Predicted')

# (1,2) Sens / Spec bar chart
ax = fig.add_subplot(gs[1, 2])
x = np.arange(2)
width = 0.3
ax.bar(x - width/2, [sens_b*100, spec_b*100], width, label='Baseline',  color='slategray')
ax.bar(x + width/2, [sens_p*100, spec_p*100], width, label='PhysREVE', color='royalblue')
ax.set_xticks(x)
ax.set_xticklabels(['Sensitivity\n(seizure recall)', 'Specificity\n(normal recall)'])
ax.set_ylabel('%')
ax.set_title('Clinical Metrics')
ax.set_ylim(0, 110)
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis='y')

plt.savefig(f'seizure_results{CFG.exp_name}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved.')


## 9  Notes & Next Steps

**What the physics pretraining provides here:**
- The leadfield bias shapes attention toward neurophysically correlated electrode pairs
- MAE + physics consistency forces the encoder to learn source-level representations
  rather than superficial signal features (useful for generalising across seizure types)
- The SNR alignment loss helps discard muscle/movement artifacts that are common
  in long-term EEG recordings (CHB-MIT includes ambulatory recordings)

**Limitations of this demo:**
- Only patient chb01 (within-subject evaluation)
- Small labelled set — cross-subject generalisation requires more data
- No temporal context across windows (windows treated independently)

**Recommended next steps:**
1. Cross-subject: pretrain on patients 1-20, fine-tune + test on patient 21-23
2. Add temporal sliding-window smoothing during inference (majority vote over 5 consecutive windows)
3. Extend to CHB-MIT full corpus or TUH EEG Seizure Corpus (TUSZ) for larger scale
4. For clinical use: optimise the decision threshold to trade sensitivity vs specificity
